# JevOps Calibration Analysis

This notebook demonstrates calibration metrics and analysis for JevOps decisions.

In [ ]:
import sys
sys.path.insert(0, '../apps/api/src')

from jevops.analytics.metrics import (
    brier_score,
    expected_calibration_error,
    calibration_curve,
    recommended_threshold,
)
from jevops.analytics.drift import population_stability_index, jensen_shannon_divergence

In [ ]:
# Example: Compute calibration metrics on synthetic data
import numpy as np

np.random.seed(42)
n = 500
true_probs = np.random.beta(2, 5, n)
outcomes = (np.random.random(n) < true_probs).astype(int)
predicted_probs = true_probs + np.random.normal(0, 0.1, n)
predicted_probs = np.clip(predicted_probs, 0, 1)

print(f'Brier Score: {brier_score(predicted_probs.tolist(), outcomes.tolist()):.4f}')
print(f'ECE: {expected_calibration_error(predicted_probs.tolist(), outcomes.tolist()):.4f}')
print(f'Recommended Threshold: {recommended_threshold(predicted_probs.tolist(), outcomes.tolist())}')

In [ ]:
# Calibration curve
bins = calibration_curve(predicted_probs.tolist(), outcomes.tolist(), n_bins=10)
for b in bins:
    if b['count'] > 0:
        print(f"Bin [{b['bin_start']:.1f}, {b['bin_end']:.1f}]: "
              f"confidence={b['avg_confidence']:.3f}, "
              f"accuracy={b['avg_accuracy']:.3f}, "
              f"n={b['count']}")

In [ ]:
# Drift detection
reference = np.random.normal(0.5, 0.15, 200).tolist()
current = np.random.normal(0.55, 0.18, 200).tolist()

psi = population_stability_index(reference, current)
jsd = jensen_shannon_divergence(reference, current)

print(f'PSI: {psi:.4f} ({"significant" if psi > 0.25 else "moderate" if psi > 0.1 else "stable"})')
print(f'JSD: {jsd:.4f}')